# 02 — Ingestão para o Data Lake
## Pipeline ETL End-to-End — Olist E-commerce Dataset

Etapa de ingestão: lê os CSVs brutos e salva em formato Parquet
na camada RAW do Data Lake (simulando AWS S3).

**Camadas do Data Lake:**
- `raw/` → dados brutos em Parquet, sem transformação
- `trusted/` → dados limpos e tipados (próximo notebook)
- `refined/` → dados agregados prontos para o DW (próximo notebook)

In [1]:
!pip install kagglehub -q

import kagglehub
import pandas as pd
import os
import json
from datetime import datetime

path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# Caminho do Data Lake local (simulando S3)
LAKE_PATH = "/content/data_lake"
RAW_PATH  = f"{LAKE_PATH}/raw"

# Criar estrutura de pastas
for folder in [RAW_PATH]:
    os.makedirs(folder, exist_ok=True)

print("✅ Ambiente configurado")
print(f"   Data Lake: {LAKE_PATH}")
print(f"   Camada RAW: {RAW_PATH}")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
✅ Ambiente configurado
   Data Lake: /content/data_lake
   Camada RAW: /content/data_lake/raw


In [2]:
# Mapeamento: arquivo CSV → nome da tabela no Data Lake
TABLES = {
    "olist_customers_dataset.csv":        "customers",
    "olist_sellers_dataset.csv":          "sellers",
    "olist_orders_dataset.csv":           "orders",
    "olist_order_items_dataset.csv":      "order_items",
    "olist_order_payments_dataset.csv":   "order_payments",
    "olist_order_reviews_dataset.csv":    "order_reviews",
    "olist_products_dataset.csv":         "products",
    "olist_geolocation_dataset.csv":      "geolocation",
    "product_category_name_translation.csv": "category_translation"
}

print("📋 Tabelas mapeadas para ingestão:")
print(f"\n{'Arquivo CSV':<50} {'Tabela no Lake'}")
print("-" * 70)
for csv, table in TABLES.items():
    print(f"   {csv:<48} → {table}")

📋 Tabelas mapeadas para ingestão:

Arquivo CSV                                        Tabela no Lake
----------------------------------------------------------------------
   olist_customers_dataset.csv                      → customers
   olist_sellers_dataset.csv                        → sellers
   olist_orders_dataset.csv                         → orders
   olist_order_items_dataset.csv                    → order_items
   olist_order_payments_dataset.csv                 → order_payments
   olist_order_reviews_dataset.csv                  → order_reviews
   olist_products_dataset.csv                       → products
   olist_geolocation_dataset.csv                    → geolocation
   product_category_name_translation.csv            → category_translation


In [3]:
def ingest_table(csv_filename, table_name, source_path, dest_path):
    """
    Lê um CSV bruto e salva como Parquet na camada RAW do Data Lake.
    Registra metadados de ingestão para rastreabilidade.
    """
    # Ler CSV
    df = pd.read_csv(f"{source_path}/{csv_filename}")

    # Adicionar metadados de ingestão
    df['_ingested_at'] = datetime.now().isoformat()
    df['_source_file'] = csv_filename

    # Criar pasta da tabela
    table_path = f"{dest_path}/{table_name}"
    os.makedirs(table_path, exist_ok=True)

    # Salvar como Parquet
    output_file = f"{table_path}/data.parquet"
    df.to_parquet(output_file, index=False)

    # Calcular tamanho dos arquivos
    csv_size  = os.path.getsize(f"{source_path}/{csv_filename}") / 1024
    parq_size = os.path.getsize(output_file) / 1024
    compression = (1 - parq_size/csv_size) * 100

    return {
        "table":        table_name,
        "rows":         len(df),
        "columns":      len(df.columns),
        "csv_kb":       round(csv_size, 1),
        "parquet_kb":   round(parq_size, 1),
        "compression":  round(compression, 1)
    }

print("✅ Função de ingestão definida")

✅ Função de ingestão definida


In [4]:
print("=" * 65)
print("INICIANDO INGESTÃO — CAMADA RAW")
print("=" * 65)

results = []
total_rows = 0

for csv_file, table_name in TABLES.items():
    result = ingest_table(csv_file, table_name, path, RAW_PATH)
    results.append(result)
    total_rows += result['rows']
    print(f"✅ {table_name:<25} {result['rows']:>8,} linhas  "
          f"({result['csv_kb']:>7.1f} KB → {result['parquet_kb']:>6.1f} KB  "
          f"compressão: {result['compression']:>5.1f}%)")

print(f"\n{'='*65}")
print(f"✅ INGESTÃO CONCLUÍDA")
print(f"   Tabelas ingeridas: {len(results)}")
print(f"   Total de linhas:   {total_rows:,}")
print(f"{'='*65}")

INICIANDO INGESTÃO — CAMADA RAW
✅ customers                   99,441 linhas  ( 8822.2 KB → 6925.8 KB  compressão:  21.5%)
✅ sellers                      3,095 linhas  (  170.6 KB →  133.0 KB  compressão:  22.0%)
✅ orders                      99,441 linhas  (17241.1 KB → 10616.4 KB  compressão:  38.4%)
✅ order_items                112,650 linhas  (15076.8 KB → 6456.7 KB  compressão:  57.2%)
✅ order_payments             103,886 linhas  ( 5641.7 KB → 3783.9 KB  compressão:  32.9%)
✅ order_reviews               99,224 linhas  (14113.0 KB → 9473.7 KB  compressão:  32.9%)
✅ products                    32,951 linhas  ( 2323.7 KB → 1394.9 KB  compressão:  40.0%)
✅ geolocation               1,000,163 linhas  (59837.8 KB → 16974.4 KB  compressão:  71.6%)
✅ category_translation            71 linhas  (    2.6 KB →    5.5 KB  compressão: -115.2%)

✅ INGESTÃO CONCLUÍDA
   Tabelas ingeridas: 9
   Total de linhas:   1,550,922


In [5]:
# Salvar log de ingestão como JSON — rastreabilidade do pipeline
log = {
    "pipeline":     "olist-etl-pipeline",
    "stage":        "ingestion",
    "executed_at":  datetime.now().isoformat(),
    "tables":       results,
    "total_rows":   total_rows
}

log_path = f"{LAKE_PATH}/logs"
os.makedirs(log_path, exist_ok=True)

with open(f"{log_path}/ingestion_log.json", "w") as f:
    json.dump(log, f, indent=2)

print("✅ Log de ingestão salvo em: /content/data_lake/logs/ingestion_log.json")
print(json.dumps(log, indent=2))

✅ Log de ingestão salvo em: /content/data_lake/logs/ingestion_log.json
{
  "pipeline": "olist-etl-pipeline",
  "stage": "ingestion",
  "executed_at": "2026-03-09T19:33:32.556007",
  "tables": [
    {
      "table": "customers",
      "rows": 99441,
      "columns": 7,
      "csv_kb": 8822.2,
      "parquet_kb": 6925.8,
      "compression": 21.5
    },
    {
      "table": "sellers",
      "rows": 3095,
      "columns": 6,
      "csv_kb": 170.6,
      "parquet_kb": 133.0,
      "compression": 22.0
    },
    {
      "table": "orders",
      "rows": 99441,
      "columns": 10,
      "csv_kb": 17241.1,
      "parquet_kb": 10616.4,
      "compression": 38.4
    },
    {
      "table": "order_items",
      "rows": 112650,
      "columns": 9,
      "csv_kb": 15076.8,
      "parquet_kb": 6456.7,
      "compression": 57.2
    },
    {
      "table": "order_payments",
      "rows": 103886,
      "columns": 7,
      "csv_kb": 5641.7,
      "parquet_kb": 3783.9,
      "compression": 32.9
    },
 

In [6]:
print("📁 ESTRUTURA DO DATA LAKE — CAMADA RAW")
print("=" * 50)

total_parquet_kb = 0
for table in sorted(os.listdir(RAW_PATH)):
    table_path = f"{RAW_PATH}/{table}"
    parquet_file = f"{table_path}/data.parquet"
    size = os.path.getsize(parquet_file) / 1024
    total_parquet_kb += size

    # Validar leitura do parquet
    df_check = pd.read_parquet(parquet_file)
    print(f"   raw/{table}/data.parquet")
    print(f"      └── {len(df_check):,} linhas · "
          f"{len(df_check.columns)} colunas · "
          f"{size:.1f} KB")

print(f"\n   Total no Data Lake: {total_parquet_kb:.1f} KB "
      f"({total_parquet_kb/1024:.1f} MB)")
print(f"\n{'='*50}")
print("PRÓXIMA ETAPA: 03_processing.ipynb")
print("Transformação e limpeza com PySpark")
print("=" * 50)

📁 ESTRUTURA DO DATA LAKE — CAMADA RAW
   raw/category_translation/data.parquet
      └── 71 linhas · 4 colunas · 5.5 KB
   raw/customers/data.parquet
      └── 99,441 linhas · 7 colunas · 6925.8 KB
   raw/geolocation/data.parquet
      └── 1,000,163 linhas · 7 colunas · 16974.4 KB
   raw/order_items/data.parquet
      └── 112,650 linhas · 9 colunas · 6456.7 KB
   raw/order_payments/data.parquet
      └── 103,886 linhas · 7 colunas · 3783.9 KB
   raw/order_reviews/data.parquet
      └── 99,224 linhas · 9 colunas · 9473.7 KB
   raw/orders/data.parquet
      └── 99,441 linhas · 10 colunas · 10616.4 KB
   raw/products/data.parquet
      └── 32,951 linhas · 11 colunas · 1394.9 KB
   raw/sellers/data.parquet
      └── 3,095 linhas · 6 colunas · 133.0 KB

   Total no Data Lake: 55764.1 KB (54.5 MB)

PRÓXIMA ETAPA: 03_processing.ipynb
Transformação e limpeza com PySpark
